# WACV — 在 Google Colab 上跑 E29 校準與 E30 主網格

依序執行，**不要跳過第 5 節的計時探測**。Colab 每次配到的 GPU 不同
（T4 / L4 / A100），E27 在 H100 上量到的「2 小時／4.2 小時」不能直接套用。
探測約 3–12 分鐘，換來的是知道這個 runtime 到底跑不跑得完。

順序（來自 `docs/NEXT_SESSION.md` §5–§6，不可重排）：

1. 確認拿到 GPU
2. 取出 token、複製 repo、確認推得上去
3. 裝套件、預抓 SD v1.4 權重、跑測試
4. 計時探測 → 決定要不要往下走
5. **E29 校準** → 綁定者診斷必須每一格都是「LPIPS hinge」
6. 判定通過才開 **E30 主網格**
7. 推上 origin、關掉 runtime

每一段實驗跑完就推一次。`/content` 在 runtime 結束時清空，沒推上去的
`runs/` 等於沒跑過。

## 1. 確認拿到 GPU

E27 實測峰值顯示記憶體 10.3 GB（`runs/e27d_C_lr0.3/summary.csv` 的 `peak_mb`），
所以 16 GB 的 T4 放得下、CPU runtime 完全跑不動。先看清楚配到什麼。

In [ ]:
!nvidia-smi

## 2. Token、複製 repo、確認推得上去

token 放在 Colab 左側的鑰匙圖示（Secrets），名稱 `GH_TOKEN`，並打開這個
notebook 的存取權。**不要貼在儲存格裡**：notebook 會把儲存格內容與輸出一起
存檔，貼進去的 token 會跟著留在檔案裡。

`git push --dry-run` 在這裡就先跑一次。網格跑完幾小時後才發現沒有寫入權限的話,
資料留在會被清空的 `/content` 裡。

In [ ]:
import importlib.util, os, subprocess, sys

BRANCH = "claude/e20-fidelity-constraint"
REPO = "github.com/Nelson0314/Non-Additive-Adversarial-Image-Editing-Defense"
WORK = "/content/WACV"

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
else:
    import getpass
    TOKEN = getpass.getpass("GitHub token: ")


def run_hidden(cmd, **kw):
    """執行並在出錯時把 token 從訊息裡遮掉再顯示。

    不用 check=True 直接讓例外冒出來：git 的錯誤訊息有時會含帶 token 的
    remote URL，而例外的 traceback 會被存進 notebook。
    """
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        out = (r.stdout + r.stderr).replace(TOKEN, "***")
        raise SystemExit(f"指令失敗：{' '.join(cmd[:2])}\n{out}")
    return r.stdout


os.chdir("/content")
if not os.path.isdir(WORK):
    # --filter=blob:none：保留完整 ref 歷史（推送才不會遇到 shallow 的限制），
    # 但只下載目前這個 commit 的檔案內容。runs/ 已累積 4000 多張 PNG，
    # 完整複製要拉幾百 MB 而這裡一個都用不到。
    run_hidden(["git", "clone", "--filter=blob:none", "--branch", BRANCH,
                f"https://{TOKEN}@{REPO}", WORK])
os.chdir(WORK)
run_hidden(["git", "config", "user.name", "Nelson0314"])
run_hidden(["git", "config", "user.email", "getyou318@gmail.com"])
run_hidden(["git", "push", "--dry-run", "origin", BRANCH])

# 驅動腳本的 PY 預設值是 Lightning AI 的 conda 路徑（`${PY:-...}`），Colab 上
# 不存在。指到目前 kernel 的直譯器，套件才會與 colab_setup.sh 裝的那一份一致。
os.environ["PY"] = sys.executable
print("PY =", sys.executable)
print("repo 就緒，分支", BRANCH, "，寫入權限已確認")
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

### 推送的輔助函式

每段實驗結束就呼叫一次。順帶核對 `.gitignore`：`runs/` 的規則曾經靜默漏掉
273 個結果檔（commit `1942e38`），而這台機器會被刪掉，漏掉就是永久遺失。
被忽略的檔案只允許 `.npy`（殘差陣列，體積大且可由 PNG 以外的產物重算）。

In [ ]:
def check_ignored():
    """確認 runs/ 底下沒有 .npy 以外的檔案被 .gitignore 排除。

    用 --ignored=matching 而非預設的 traditional：後者會把整個被忽略的目錄
    收合成一行，逐圖子目錄裡混著要入庫的 PNG 與不入庫的 .npy 時看不出實情。
    """
    out = subprocess.run(
        ["git", "status", "--porcelain", "--ignored=matching", "runs/"],
        capture_output=True, text=True).stdout
    bad = [ln[3:] for ln in out.splitlines()
           if ln.startswith("!!") and not ln.endswith(".npy")]
    if bad:
        raise SystemExit("這些結果檔被 .gitignore 排除了，先修 .gitignore：\n"
                         + "\n".join(bad[:20]))
    print("  .gitignore 檢查通過（只有 .npy 被排除）")


def push(msg):
    # 先 add 再檢查：檢查的是「加完之後還有什麼被擋在外面」，順序反過來會把
    # 尚未 add 的正常新檔也算進去。
    subprocess.run(["git", "add", "-A", "runs/"], check=True)
    check_ignored()
    if subprocess.run(["git", "diff", "--cached", "--quiet"]).returncode == 0:
        print("  沒有新的結果檔可提交")
        return
    subprocess.run(["git", "commit", "-m", msg], check=True)
    run_hidden(["git", "push", "origin", BRANCH])
    print("  已推上 origin：", msg)

## 3. 環境

`colab_setup.sh` 刻意**不安裝 torch／torchvision**：Colab 的映像檔已裝好與該
runtime 驅動相符的版本，從 PyPI 覆蓋掉會讓 `torch.cuda.is_available()` 變成
False。它把現有版本寫成 pip constraint 擋住升級，裝完再核對一次。

同時預抓 SD v1.4（4 GB）並跑一次測試，基準 **253 passed / 1 skipped**。
約 5–10 分鐘。

In [ ]:
!bash scripts/drivers/colab_setup.sh
# `!` 失敗不會在 notebook 裡拋例外，只是印完就往下走。這裡把離開碼轉成
# 中止：環境沒裝好而繼續跑 E29，會在幾分鐘後以完全無關的錯誤現形。
assert _exit_code == 0, "colab_setup.sh 失敗，看上面的輸出，不要往下跑"

## 4. 計時探測

跑兩個步數不同的極短 run，相減消掉每格的固定成本，斜率就是每步成本；再用一個
開評測的 run 量評測成本。輸出會給出 E29 與 E30 的推算時間，以及記憶體與連線
上限的判定。

比較基準是 E27 的 H100 實測：每步 2.47 s、評測 41.4 s、峰值 10.3 GB。

In [ ]:
# 經 bash 而非 IPython 預設的 sh：Ubuntu 的 /bin/sh 是 dash，不支援
# pipefail，離開碼會是 tee 的（恆為 0），探測失敗會被吃掉。
!bash -c 'set -o pipefail; python scripts/colab_probe.py 2>&1 | tee runs/logs/colab_probe.log'
assert _exit_code == 0, "colab_probe.py 失敗"

**讀完再往下。** 若判定出現「單次呼叫上限超過 3 小時」或「上限情境超過
11 小時」，先換 runtime（Colab Pro 的 A100／L4）或改為只跑 τ=0.05 一組，不要
直接開 E30——中途斷線的話 `run_defense.py` 沒有續跑旗標，該次呼叫要整個重跑。

In [ ]:
push("Record the Colab timing probe")

## 5. E29 校準

8 格、60 步、不評測。site C 的 lr 掃 0.1／0.3，site P 掃 0.03／0.1。

要重跑校準的理由：E27 定出的 lr 是在**還沒有色度約束**的程式上量的，而 site C
在舊約束下的解色度偏壓 4.97、門檻 0.8，加上約束後一定會被壓下來。壓下來之後
lr=0.3 還適不適用是未知的。

In [ ]:
!bash scripts/drivers/e29_calibration.sh
assert _exit_code == 0, "E29 校準失敗，不要 push 半套資料"

In [ ]:
push("Add the E29 calibration under the three-constraint set")

## 6. 判定

看上一段結尾的綁定者診斷表。**每一格的判定都必須是「LPIPS hinge」**，不是
硬上界、不是防禦 margin、不是色度 hinge。

判定不通過時分兩種情況（`docs/NEXT_SESSION.md` §5）：

- 降 lr 之後末色度掉到 0.8 以下、且 LPIPS hinge 接手 → 改用該 lr 開 E30。
- 降 lr 之後色度仍是綁定者，或末色度只是隨 `edit_shift` 一起塌掉（防禦能力歸零
  換來合規）→ 那代表色度偏壓是 site C 這個參數化的固有代價，不是學習率的問題。
  此時**不要開 E30**，回報使用者，改換一個非加性參數化。

通過的話把定出的兩個 lr 填進下一格。

In [ ]:
LR_C = "0.3"    # ← 依 E29 的診斷結果修改
LR_P = "0.03"   # ← 依 E29 的診斷結果修改
print("E30 將使用 LR_C =", LR_C, " LR_P =", LR_P)

## 7. E30 主網格

2 site × 3 τ × 6 圖 = 36 格，`--stop_on_plateau` 開啟，上限 150 步。
τ 的順序是 0.05／0.02／0.10：0.05 是與 E21／E23 對應的格子，機時不足時
至少有關鍵比較。

跑完自動接彙整與綁定者診斷。

In [ ]:
os.environ["LR_C"], os.environ["LR_P"] = LR_C, LR_P
!bash scripts/drivers/e30_grid.sh
# 這裡不像前面幾格那樣中止。網格中途失敗時已完成的格子仍是證據，
# 而這台機器會被刪掉——先推上去再查原因。
if _exit_code != 0:
    print("\n[!] e30_grid.sh 以離開碼", _exit_code,
          "結束。仍執行下一格把已完成的部分推上 origin，再回頭查原因。")

In [ ]:
push(f"Add the E30 main grid (lr_C={LR_C}, lr_P={LR_P})")

## 8. 收尾

確認上一格印出「已推上 origin」之後，**關掉 runtime**（Runtime → Disconnect
and delete runtime）。`/content` 會被清空，沒推上去的東西拿不回來。

若中途斷線：重跑第 2、3 節（複製 repo、裝套件），已經推上去的 run 目錄會
一起回來，只需重跑尚未完成的那幾次呼叫。`run_defense.py` 沒有續跑旗標，
被中斷的那個 run 目錄要整個重跑。

In [ ]:
print(subprocess.run(["git", "log", "--oneline", "-5"],
                     capture_output=True, text=True).stdout)
print(subprocess.run(["git", "status", "--short", "--branch"],
                     capture_output=True, text=True).stdout)
print("確認 origin 已同步之後即可關掉 runtime。")